# Full Pipeline — KNN Anonymization (With Target)

Self-contained batch notebook. Set `DATA_FILE`, parameter lists, and batch controls in **Configuration**; schema is inferred automatically when data is loaded.

Supports **binary**, **multi-class categorical**, and **continuous numeric** targets (set `TARGET_COL_OVERRIDE` / `TARGET_KIND_OVERRIDE` when auto-detection is not enough).

**Distance modes:** `weighted_sum` (separate num + cat metrics) or `gower` (mixed-type Gower on raw numeric ranges + category mismatch).

**Target synthesis** is controlled separately from feature categoricals via `TARGET_GEN_METHOD`:
- **binary / multiclass** → `mode`, `weighted_mode`, or `probability` (defaults to `cat_gen_method` when set to `None`)
- **numeric** → automatically uses `num_gen_method` (`interpolation` or `weighted_mean`)

**Utility metrics by target kind:**
- **binary** → AUC / F1 retention (LogisticRegression on real vs synthetic train)
- **multiclass** → macro F1 retention
- **numeric** → R² and RMSE retention (Ridge regression on real vs synthetic train)

Column skipping follows the Karabo `others_discoveries` approach: drop fully empty columns, optionally exclude manual columns or constant features via `EXCLUDE_COLS_OVERRIDE` / `SKIP_CONSTANT_COLS`.

Each experiment run can write a folder under `experiments/` (`SAVE_EXPERIMENT_ARTIFACTS`) containing the anonymized dataset, `parameters.json`, metric CSVs, and `summary.json`. A consolidated `experiment_ranking.csv` is always written in this folder.


In [1]:
# --- imports ---
import itertools
import json
import time
import tracemalloc
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler, StandardScaler

ROW_SYNTHESIS_MODE = "donor"
MISSING_LABEL = "Missing"
TVD_THRESHOLD = 0.10
KS_THRESHOLD = 0.10
PASS_RATE_TARGET = 0.85
RANDOM_STATE = 42
EXPERIMENT_RANKING_CSV = "experiment_ranking.csv"

DISCRETE_TARGET_GEN_METHODS = ("mode", "weighted_mode", "probability")
NUMERIC_TARGET_GEN_METHODS = ("interpolation", "weighted_mean")
CAT_DISTANCE_METRICS = ("hamming", "jaccard", "overlap", "cosine")
DISTANCE_MODES = ("weighted_sum", "gower")


## Configuration


In [2]:
# Data
DATA_FILE = "../Bank Customer Churn Prediction.csv"

# Optional schema overrides (None = auto-detect from CSV)
TARGET_COL_OVERRIDE = "estimated_salary"
TARGET_KIND_OVERRIDE = None  # None | "binary" | "multiclass" | "numeric"
CAT_TARGET_MAX_UNIQUE = 20   # numeric columns with <= this many uniques → discrete target
ID_COLS_OVERRIDE = None

# Column skipping (aligned with others_discoveries Karabo notebooks)
EXCLUDE_COLS_OVERRIDE = None   # e.g. ["Surname", "leaky_col"] — kept in df, excluded from pipeline
SKIP_CONSTANT_COLS = False     # True → skip features with <=1 unique non-missing value
DROP_FULLY_EMPTY_COLS = True     # True → drop columns that are 100% missing (Karabo default)

# Batch controls
BATCH_START = 0
BATCH_LIMIT = None  # None = run full grid
CHECKPOINT_EVERY = 25

# Per-experiment artifacts (anonymized CSV, parameters JSON, metrics, etc.)
SAVE_EXPERIMENT_ARTIFACTS = True
EXPERIMENT_OUTPUT_DIR = "experiments"
SKIP_EXISTING_EXPERIMENTS = True  # skip configs already present in experiment_ranking.csv

# Parameter lists (full cartesian product)
K_ANONYMITY = [3]
K_NEIGHBORS = [3, 15]
CAT_GEN_METHOD = ["mode", "weighted_mode", "probability"]
NUM_GEN_METHOD = ["interpolation", "weighted_mean"]
# Discrete target synthesis (binary/multiclass). None = follow cat_gen_method.
# For numeric targets this grid dimension is dropped after load (num_gen_method drives target).
TARGET_GEN_METHOD = [None, "probability"]
SCALER_METHOD = [ "robust"]
DISTANCE_MODE = ["weighted_sum", "gower"]  # gower = mixed-type Gower distance (cat_distance_metric ignored)
NUM_DISTANCE_METRIC = ["euclidean"]
MINKOWSKI_P = [3]
CAT_DISTANCE_METRIC = ["hamming","jaccard","cosine"]  # hamming | jaccard | overlap | cosine
NUM_WEIGHT = [1.0]
CAT_WEIGHT = [1.0]
DISTANCE_PROFILE = ["balanced"]

PARAMETER_COMBINATIONS = pd.DataFrame([
    {
        "k_anonymity": k,
        "k_neighbors": kn,
        "cat_gen_method": cat,
        "num_gen_method": num,
        "target_gen_method": tgt,
        "scaler_method": scale,
        "distance_mode": dmode,
        "num_distance_metric": nmet,
        "minkowski_p": mp,
        "cat_distance_metric": cmet,
        "num_weight": nw,
        "cat_weight": cw,
        "distance_profile": dprof,
    }
    for k, kn, cat, num, tgt, scale, dmode, nmet, mp, cmet, nw, cw, dprof in itertools.product(
        K_ANONYMITY,
        K_NEIGHBORS,
        CAT_GEN_METHOD,
        NUM_GEN_METHOD,
        TARGET_GEN_METHOD,
        SCALER_METHOD,
        DISTANCE_MODE,
        NUM_DISTANCE_METRIC,
        MINKOWSKI_P,
        CAT_DISTANCE_METRIC,
        NUM_WEIGHT,
        CAT_WEIGHT,
        DISTANCE_PROFILE,
    )
])

print("Grid (before target-kind adjustment):", len(PARAMETER_COMBINATIONS), "configs")
PARAMETER_COMBINATIONS.head()


Grid (before target-kind adjustment): 144 configs


,k_anonymity,k_neighbors,cat_gen_method,num_gen_method,target_gen_method,scaler_method,distance_mode,num_distance_metric,minkowski_p,cat_distance_metric,num_weight,cat_weight,distance_profile
0,3,3,mode,interpolation,NaN,robust,weighted_sum,euclidean,3,hamming,1.0,1.0,balanced
1,3,3,mode,interpolation,NaN,robust,weighted_sum,euclidean,3,jaccard,1.0,1.0,balanced
2,3,3,mode,interpolation,NaN,robust,weighted_sum,euclidean,3,cosine,1.0,1.0,balanced
3,3,3,mode,interpolation,NaN,robust,gower,euclidean,3,hamming,1.0,1.0,balanced
4,3,3,mode,interpolation,NaN,robust,gower,euclidean,3,jaccard,1.0,1.0,balanced


## Metrics


In [3]:
PSI_THRESHOLD = 0.25
RARE_CATEGORY_THRESHOLD = 0.05
TARGET_RATE_DRIFT_THRESHOLD = 0.05
TARGET_KS_THRESHOLD = 0.10
AUC_RETENTION_THRESHOLD = 0.80
F1_RETENTION_THRESHOLD = 0.80
R2_RETENTION_THRESHOLD = 0.80
RMSE_RETENTION_THRESHOLD = 0.80


def calculate_psi(real_values, synth_values, bins: int = 10) -> float:
    real_values = pd.Series(real_values).dropna().astype(float)
    synth_values = pd.Series(synth_values).dropna().astype(float)
    if len(real_values) == 0 or len(synth_values) == 0:
        return float("nan")
    if real_values.nunique() <= 1:
        return 0.0 if synth_values.nunique() <= 1 and synth_values.iloc[0] == real_values.iloc[0] else float("nan")
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(real_values, quantiles))
    if len(edges) < 3:
        return float("nan")
    real_counts, _ = np.histogram(real_values, bins=edges)
    synth_counts, _ = np.histogram(synth_values, bins=edges)
    real_pct = real_counts / max(1, real_counts.sum())
    synth_pct = synth_counts / max(1, synth_counts.sum())
    epsilon = 1e-6
    return float(np.sum((synth_pct - real_pct) * np.log((synth_pct + epsilon) / (real_pct + epsilon))))


def categorical_distribution_metrics(real_data, synth_data, cat_cols):
    rows = []
    for col in cat_cols:
        real_dist = real_data[col].astype(str).value_counts(normalize=True, dropna=False)
        synth_dist = synth_data[col].astype(str).value_counts(normalize=True, dropna=False)
        all_categories = sorted(set(real_dist.index) | set(synth_dist.index))
        real_aligned = real_dist.reindex(all_categories, fill_value=0)
        synth_aligned = synth_dist.reindex(all_categories, fill_value=0)
        drift = float(0.5 * np.abs(real_aligned - synth_aligned).sum())
        real_categories = set(real_dist.index)
        synth_categories = set(synth_dist.index)
        rows.append({
            "column": col,
            "TVD": round(drift, 4),
            "pass_tvd": drift < TVD_THRESHOLD,
            "real_category_count": len(real_categories),
            "synthetic_category_count": len(synth_categories),
            "category_coverage": round(len(real_categories & synth_categories) / max(1, len(real_categories)), 4),
            "new_synthetic_category_count": len(synth_categories - real_categories),
            "missing_synthetic_category_count": len(real_categories - synth_categories),
            "max_category_proportion_diff": round(float(np.abs(real_aligned - synth_aligned).max()), 4),
        })
    return pd.DataFrame(rows)


def rare_category_metrics(original_df, synthetic_df, cols, threshold=RARE_CATEGORY_THRESHOLD):
    rows = []
    for col in cols:
        orig_freq = original_df[col].value_counts(normalize=True)
        rare_cats = orig_freq[orig_freq < threshold].index
        if len(rare_cats) == 0:
            rows.append({
                "column": col,
                "rare_category_count": 0,
                "original_rare_mass": 0.0,
                "synthetic_rare_mass": 0.0,
                "retention_ratio": float("nan"),
            })
            continue
        synth_freq = synthetic_df[col].value_counts(normalize=True)
        orig_mass = float(orig_freq.loc[rare_cats].sum())
        synth_mass = float(synth_freq.reindex(rare_cats, fill_value=0).sum())
        rows.append({
            "column": col,
            "rare_category_count": int(len(rare_cats)),
            "original_rare_mass": round(orig_mass, 4),
            "synthetic_rare_mass": round(synth_mass, 4),
            "retention_ratio": round(synth_mass / orig_mass, 4) if orig_mass > 0 else float("nan"),
        })
    return pd.DataFrame(rows)


def numerical_distribution_metrics(real_data, synth_data, num_cols):
    rows = []
    for col in num_cols:
        real = pd.to_numeric(real_data[col], errors="coerce").dropna()
        synth = pd.to_numeric(synth_data[col], errors="coerce").dropna()
        if len(real) == 0 or len(synth) == 0:
            continue
        ks_stat = float(ks_2samp(real, synth).statistic) if real.nunique() > 1 and synth.nunique() > 1 else float("nan")
        psi = calculate_psi(real, synth)
        rows.append({
            "column": col,
            "KS_statistic": round(ks_stat, 4) if pd.notna(ks_stat) else np.nan,
            "pass_ks": ks_stat < KS_THRESHOLD if pd.notna(ks_stat) else False,
            "psi": round(psi, 4) if pd.notna(psi) else np.nan,
            "pass_psi": psi < PSI_THRESHOLD if pd.notna(psi) else False,
            "real_mean": round(float(real.mean()), 4),
            "synthetic_mean": round(float(synth.mean()), 4),
            "mean_abs_diff": round(abs(float(real.mean() - synth.mean())), 4),
            "real_median": round(float(real.median()), 4),
            "synthetic_median": round(float(synth.median()), 4),
            "median_abs_diff": round(abs(float(real.median() - synth.median())), 4),
            "real_std": round(float(real.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "std_abs_diff": round(abs(float(real.std() - synth.std())), 4),
        })
    return pd.DataFrame(rows)


def cramers_v(x, y) -> float:
    tbl = pd.crosstab(x.astype(str), y.astype(str))
    if tbl.shape[0] < 2 or tbl.shape[1] < 2:
        return float("nan")
    chi2 = chi2_contingency(tbl)[0]
    n = tbl.sum().sum()
    k = min(tbl.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if n > 0 and k > 0 else 0.0


def _discretize_for_association(series: pd.Series, max_bins: int = 10) -> pd.Series:
    nums = pd.to_numeric(series, errors="coerce")
    if nums.notna().mean() >= 0.5 and nums.nunique(dropna=True) > max_bins:
        ranked = nums.rank(method="first")
        return pd.qcut(ranked, q=max_bins, duplicates="drop").astype(str)
    return series.astype(str).fillna(MISSING_LABEL)


def _feature_target_strength(df, feature_col, target_col, target_kind) -> float:
    if target_kind == "numeric" and pd.api.types.is_numeric_dtype(df[feature_col]):
        x = pd.to_numeric(df[feature_col], errors="coerce")
        y = pd.to_numeric(df[target_col], errors="coerce")
        val = x.corr(y)
        return float(val) if pd.notna(val) else float("nan")
    x = _discretize_for_association(df[feature_col]) if pd.api.types.is_numeric_dtype(df[feature_col]) else df[feature_col]
    return cramers_v(x, df[target_col])


def target_relationship_metrics(df_actual, df_out, feature_cols, target_col, target_kind):
    rows = []
    for col in feature_cols:
        if col == target_col:
            continue
        metric_name = "correlation" if target_kind == "numeric" and pd.api.types.is_numeric_dtype(df_actual[col]) else "cramers_v"
        a = _feature_target_strength(df_actual, col, target_col, target_kind)
        s = _feature_target_strength(df_out, col, target_col, target_kind)
        if pd.notna(a) and pd.notna(s):
            rows.append({
                "label": target_col,
                "column": col,
                "metric": metric_name,
                "actual": round(float(a), 4),
                "synthetic": round(float(s), 4),
                "drift": round(abs(float(a) - float(s)), 4),
            })
    return pd.DataFrame(rows)


def numerical_correlation_metrics(real_data, synth_data, num_cols):
    if len(num_cols) < 2:
        return pd.DataFrame()
    real_corr = real_data[num_cols].astype(float).corr()
    synth_corr = synth_data[num_cols].astype(float).corr()
    rows = []
    for c1, c2 in itertools.combinations(num_cols, 2):
        real_val = real_corr.loc[c1, c2]
        synth_val = synth_corr.loc[c1, c2]
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_corr": round(float(real_val), 4) if pd.notna(real_val) else np.nan,
            "synthetic_corr": round(float(synth_val), 4) if pd.notna(synth_val) else np.nan,
            "abs_corr_diff": round(abs(float(real_val) - float(synth_val)), 4)
            if pd.notna(real_val) and pd.notna(synth_val) else np.nan,
        })
    return pd.DataFrame(rows)


def categorical_pair_relationship_metrics(real_data, synth_data, cat_cols):
    rows = []
    for c1, c2 in itertools.combinations(cat_cols, 2):
        real_v = cramers_v(real_data[c1], real_data[c2])
        synth_v = cramers_v(synth_data[c1], synth_data[c2])
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_cramers_v": round(real_v, 4) if pd.notna(real_v) else np.nan,
            "synthetic_cramers_v": round(synth_v, 4) if pd.notna(synth_v) else np.nan,
            "abs_cramers_v_diff": round(abs(real_v - synth_v), 4) if pd.notna(real_v) and pd.notna(synth_v) else np.nan,
        })
    return pd.DataFrame(rows)


def categorical_numerical_relationship_metrics(real_data, synth_data, cat_cols, num_cols, max_categories=25):
    rows = []
    for cat in cat_cols:
        if real_data[cat].nunique(dropna=False) > max_categories:
            continue
        for num in num_cols:
            real_group = real_data.groupby(cat)[num].mean()
            synth_group = synth_data.groupby(cat)[num].mean()
            common_groups = sorted(set(real_group.index) & set(synth_group.index))
            if not common_groups:
                continue
            real_values = real_group.reindex(common_groups)
            synth_values = synth_group.reindex(common_groups)
            scale = real_data[num].astype(float).std()
            if pd.isna(scale) or scale == 0:
                scale = 1.0
            mean_group_diff = float(np.mean(np.abs(real_values - synth_values)) / scale)
            rows.append({
                "categorical_column": cat,
                "numerical_column": num,
                "common_category_count": len(common_groups),
                "normalised_group_mean_diff": round(mean_group_diff, 4),
            })
    return pd.DataFrame(rows)


def calc_iv(df, feature, target):
    eps = 1e-6
    target_num = pd.to_numeric(df[target], errors="coerce")
    total_good = int((target_num == 0).sum())
    total_bad = int((target_num == 1).sum())
    iv = 0.0
    for _, sub in df.groupby(feature, dropna=False):
        y = pd.to_numeric(sub[target], errors="coerce")
        good = int((y == 0).sum())
        bad = int((y == 1).sum())
        dist_good = good / max(total_good, 1)
        dist_bad = bad / max(total_bad, 1)
        woe = np.log((dist_good + eps) / (dist_bad + eps))
        iv += (dist_good - dist_bad) * woe
    return float(iv)


def iv_retention_metrics(original_df, synthetic_df, cols, target, target_kind):
    if target_kind != "binary":
        return pd.DataFrame([{"status": "skipped_iv_binary_only"}])
    rows = []
    for col in cols:
        orig_iv = calc_iv(original_df, col, target)
        synth_iv = calc_iv(synthetic_df, col, target)
        rows.append({
            "feature": col,
            "original_iv": round(orig_iv, 4),
            "synthetic_iv": round(synth_iv, 4),
            "retention_ratio": round(synth_iv / orig_iv, 4) if abs(orig_iv) > 1e-9 else np.nan,
            "absolute_delta": round(abs(orig_iv - synth_iv), 4),
        })
    return pd.DataFrame(rows).sort_values("absolute_delta", ascending=False)


def target_distribution_metrics(df_actual, df_out, target_col, target_kind):
    if target_kind == "numeric":
        real = pd.to_numeric(df_actual[target_col], errors="coerce").dropna()
        synth = pd.to_numeric(df_out[target_col], errors="coerce").dropna()
        mean_drift = abs(float(real.mean() - synth.mean()))
        std_scale = max(float(real.std()), 1e-6)
        drift = mean_drift / std_scale
        ks_stat = (
            float(ks_2samp(real, synth).statistic)
            if len(real) and len(synth) and real.nunique() > 1 and synth.nunique() > 1
            else float("nan")
        )
        return pd.DataFrame([{
            "target_col": target_col,
            "target_kind": target_kind,
            "actual_rate": round(float(real.mean()), 4),
            "synthetic_rate": round(float(synth.mean()), 4),
            "rate_drift": round(drift, 4),
            "pass_rate_drift": drift < TARGET_RATE_DRIFT_THRESHOLD,
            "distribution_metric": "normalized_mean_drift",
            "target_ks": round(ks_stat, 4) if pd.notna(ks_stat) else None,
            "pass_target_ks": ks_stat < TARGET_KS_THRESHOLD if pd.notna(ks_stat) else None,
        }])

    real_dist = df_actual[target_col].astype(str).value_counts(normalize=True, dropna=False)
    synth_dist = df_out[target_col].astype(str).value_counts(normalize=True, dropna=False)
    all_categories = sorted(set(real_dist.index) | set(synth_dist.index))
    real_aligned = real_dist.reindex(all_categories, fill_value=0)
    synth_aligned = synth_dist.reindex(all_categories, fill_value=0)
    tvd = float(0.5 * np.abs(real_aligned - synth_aligned).sum())
    if target_kind == "binary":
        actual_rate = float(pd.to_numeric(df_actual[target_col], errors="coerce").mean())
        synthetic_rate = float(pd.to_numeric(df_out[target_col], errors="coerce").mean())
        drift = abs(actual_rate - synthetic_rate)
        metric_name = "positive_rate_drift"
    else:
        actual_rate = float(real_aligned.max()) if len(real_aligned) else float("nan")
        synthetic_rate = float(synth_aligned.max()) if len(synth_aligned) else float("nan")
        drift = tvd
        metric_name = "tvd"
    return pd.DataFrame([{
        "target_col": target_col,
        "target_kind": target_kind,
        "actual_rate": round(actual_rate, 4),
        "synthetic_rate": round(synthetic_rate, 4),
        "rate_drift": round(drift, 4),
        "pass_rate_drift": drift < TARGET_RATE_DRIFT_THRESHOLD,
        "distribution_metric": metric_name,
    }])


def privacy_metrics(df_actual, df_out, qi_cols, suppressed):
    full_hash_actual = pd.util.hash_pandas_object(df_actual[qi_cols].astype(str), index=False)
    full_hash_out = pd.util.hash_pandas_object(df_out[qi_cols].astype(str), index=False)
    replaced_idx = suppressed[suppressed].index
    replaced_out = df_out.loc[replaced_idx, qi_cols].astype(str)
    replaced_hash = pd.util.hash_pandas_object(replaced_out, index=False)
    synthetic_duplicate_rate = 1.0 - (replaced_hash.nunique() / max(1, len(replaced_hash)))
    exact_match_to_actual_rate = float(replaced_hash.isin(set(full_hash_actual)).mean())
    full_dataset_duplicate_rate = 1.0 - (full_hash_out.nunique() / max(1, len(full_hash_out)))
    replaced_feature_cols = [c for c in df_actual.columns if c not in set(globals().get("ID_COLS", []))]
    exact_match_replaced_rows = len(
        df_out.loc[replaced_idx, replaced_feature_cols].merge(df_actual[replaced_feature_cols], how="inner")
    ) / max(len(replaced_idx), 1)
    return pd.DataFrame([{
        "replaced_rows": int(len(replaced_idx)),
        "replaced_unique_profiles": int(replaced_hash.nunique()),
        "synthetic_duplicate_rate": round(synthetic_duplicate_rate, 6),
        "exact_match_to_actual_rate": round(exact_match_to_actual_rate, 6),
        "full_dataset_duplicate_rate": round(full_dataset_duplicate_rate, 6),
        "replaced_exact_match_rate": round(exact_match_replaced_rows, 6),
    }])


def _prepare_target_series(series, target_kind):
    if target_kind == "numeric":
        return pd.to_numeric(series, errors="coerce")
    if target_kind == "binary":
        return pd.to_numeric(series, errors="coerce")
    enc = LabelEncoder()
    return pd.Series(
        enc.fit_transform(series.astype(str).fillna(MISSING_LABEL)),
        index=series.index,
    )


def utility_metrics(df_actual, df_out, feature_cols, target_col, target_kind, random_state=42):
    y_all = _prepare_target_series(df_actual[target_col], target_kind)
    valid_idx = y_all.notna() if target_kind == "numeric" else pd.Series(True, index=y_all.index)
    y_all = y_all[valid_idx]
    if len(y_all) == 0:
        return pd.DataFrame([{"status": "skipped_empty_target"}])

    min_classes = 2 if target_kind != "numeric" else 1
    if target_kind != "numeric" and y_all.nunique(dropna=True) < min_classes:
        return pd.DataFrame([{"status": "skipped_single_class_target"}])

    stratify = y_all if target_kind != "numeric" and y_all.nunique(dropna=True) > 1 else None
    train_idx, test_idx = train_test_split(
        y_all.index, test_size=0.2, random_state=random_state, stratify=stratify
    )

    def encode_frame(frame):
        out = frame[feature_cols].copy()
        for col in feature_cols:
            if pd.api.types.is_numeric_dtype(out[col]):
                numeric = pd.to_numeric(out[col], errors="coerce")
                out[col] = numeric.fillna(numeric.median())
            else:
                enc = LabelEncoder()
                out[col] = enc.fit_transform(out[col].astype(str).fillna(MISSING_LABEL))
        return out

    X_train = encode_frame(df_actual.loc[train_idx])
    X_test = encode_frame(df_actual.loc[test_idx])
    X_syn_train = encode_frame(df_out.loc[train_idx])
    y_train = y_all.loc[train_idx]
    y_test = y_all.loc[test_idx]
    y_syn_train = _prepare_target_series(df_out.loc[train_idx, target_col], target_kind)

    if target_kind != "numeric":
        if y_train.nunique() < 2 or y_syn_train.nunique() < 2:
            return pd.DataFrame([{"status": "skipped_single_class_in_training"}])

    rows = []
    roc_curves = {}
    for name, X_fit, y_fit in [("real_train", X_train, y_train), ("synthetic_train", X_syn_train, y_syn_train)]:
        if target_kind == "numeric":
            model = Ridge(random_state=random_state)
            model.fit(X_fit, y_fit)
            pred = model.predict(X_test)
            rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
            r2 = float(r2_score(y_test, pred))
            rows.append({
                "training_data": name,
                "task": "regression",
                "r2": round(r2, 4),
                "rmse": round(rmse, 4),
                "auc": np.nan,
                "accuracy": np.nan,
                "f1_score": np.nan,
            })
            continue

        if target_kind == "binary":
            model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)
            model.fit(X_fit, y_fit)
            proba = model.predict_proba(X_test)[:, 1]
            pred = (proba >= 0.5).astype(int)
            fpr, tpr, _ = roc_curve(y_test, proba)
            roc_curves[name] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
            rows.append({
                "training_data": name,
                "task": "binary_classification",
                "auc": round(float(roc_auc_score(y_test, proba)), 4),
                "accuracy": round(float(accuracy_score(y_test, pred)), 4),
                "f1_score": round(float(f1_score(y_test, pred)), 4),
                "r2": np.nan,
                "rmse": np.nan,
            })
            continue

        model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)
        model.fit(X_fit, y_fit)
        pred = model.predict(X_test)
        rows.append({
            "training_data": name,
            "task": "multiclass_classification",
            "accuracy": round(float(accuracy_score(y_test, pred)), 4),
            "f1_score": round(float(f1_score(y_test, pred, average="macro")), 4),
            "auc": np.nan,
            "r2": np.nan,
            "rmse": np.nan,
        })

    result = pd.DataFrame(rows)
    real_row = result.loc[result["training_data"] == "real_train"].iloc[0]
    syn_row = result.loc[result["training_data"] == "synthetic_train"].iloc[0]
    result["auc_retention_ratio"] = np.nan
    result["f1_retention_ratio"] = np.nan
    result["r2_retention_ratio"] = np.nan
    result["rmse_retention_ratio"] = np.nan

    if target_kind == "numeric":
        real_r2 = float(real_row["r2"])
        syn_r2 = float(syn_row["r2"])
        real_rmse = float(real_row["rmse"])
        syn_rmse = float(syn_row["rmse"])
        result.loc[result["training_data"] == "synthetic_train", "r2_retention_ratio"] = round(
            syn_r2 / real_r2 if abs(real_r2) > 1e-9 else np.nan, 4
        )
        result.loc[result["training_data"] == "synthetic_train", "rmse_retention_ratio"] = round(
            real_rmse / syn_rmse if syn_rmse > 1e-9 else np.nan, 4
        )
        result.attrs["real_r2"] = real_r2
        result.attrs["synthetic_r2"] = syn_r2
        result.attrs["real_rmse"] = real_rmse
        result.attrs["synthetic_rmse"] = syn_rmse
        result.attrs["utility_retention_ratio"] = float(
            result.loc[result["training_data"] == "synthetic_train", "r2_retention_ratio"].iloc[0]
        )
    else:
        real_f1 = float(real_row["f1_score"]) if pd.notna(real_row["f1_score"]) else float("nan")
        syn_f1 = float(syn_row["f1_score"]) if pd.notna(syn_row["f1_score"]) else float("nan")
        result.loc[result["training_data"] == "synthetic_train", "f1_retention_ratio"] = round(
            syn_f1 / real_f1 if pd.notna(real_f1) and real_f1 > 1e-9 else np.nan, 4
        )
        result.attrs["real_f1"] = real_f1
        result.attrs["synthetic_f1"] = syn_f1
        if target_kind == "binary":
            real_auc = float(real_row["auc"])
            syn_auc = float(syn_row["auc"])
            result.loc[result["training_data"] == "synthetic_train", "auc_retention_ratio"] = round(
                syn_auc / real_auc if real_auc > 1e-9 else np.nan, 4
            )
            result.attrs["real_auc"] = real_auc
            result.attrs["synthetic_auc"] = syn_auc
            result.attrs["utility_retention_ratio"] = float(result.loc[result["training_data"] == "synthetic_train", "auc_retention_ratio"].iloc[0])
        else:
            result.attrs["real_auc"] = float("nan")
            result.attrs["synthetic_auc"] = float("nan")
            result.attrs["utility_retention_ratio"] = float(result.loc[result["training_data"] == "synthetic_train", "f1_retention_ratio"].iloc[0])

    result.attrs["roc_curves"] = roc_curves
    result.attrs["target_kind"] = target_kind
    return result


def _metric_val(summary, key, default):
    val = summary.get(key)
    return default if val is None else float(val)


def build_karabo_scorecard(summary):
    target_kind = summary.get("target_kind")
    rows = [
        {"area": "Karabo-Categorical", "metric": "mean_category_coverage>=1.0", "value": summary.get("mean_category_coverage"), "pass": _metric_val(summary, "mean_category_coverage", 0.0) >= 1.0},
        {"area": "Karabo-Categorical", "metric": "new_synthetic_categories==0", "value": summary.get("total_new_categories"), "pass": int(summary.get("total_new_categories") or 0) == 0},
        {"area": "Karabo-Numerical", "metric": "mean_PSI<0.25", "value": summary.get("mean_psi"), "pass": _metric_val(summary, "mean_psi", 999.0) < PSI_THRESHOLD},
        {"area": "Karabo-Target", "metric": "target_rate_drift<0.05", "value": summary.get("target_rate_drift"), "pass": _metric_val(summary, "target_rate_drift", 999.0) < TARGET_RATE_DRIFT_THRESHOLD},
    ]
    if target_kind == "numeric":
        rows.extend([
            {"area": "Karabo-Target", "metric": "target_ks<0.10", "value": summary.get("target_ks"), "pass": _metric_val(summary, "target_ks", 999.0) < TARGET_KS_THRESHOLD},
            {"area": "Karabo-Utility", "metric": "r2_retention>=0.80", "value": summary.get("r2_retention_ratio"), "pass": _metric_val(summary, "r2_retention_ratio", 0.0) >= R2_RETENTION_THRESHOLD},
            {"area": "Karabo-Utility", "metric": "rmse_retention>=0.80", "value": summary.get("rmse_retention_ratio"), "pass": _metric_val(summary, "rmse_retention_ratio", 0.0) >= RMSE_RETENTION_THRESHOLD},
        ])
    elif target_kind == "binary":
        rows.extend([
            {"area": "Karabo-Utility", "metric": "auc_retention>=0.80", "value": summary.get("auc_retention_ratio"), "pass": _metric_val(summary, "auc_retention_ratio", 0.0) >= AUC_RETENTION_THRESHOLD},
            {"area": "Karabo-Utility", "metric": "f1_retention>=0.80", "value": summary.get("f1_retention_ratio"), "pass": _metric_val(summary, "f1_retention_ratio", 0.0) >= F1_RETENTION_THRESHOLD},
        ])
    elif target_kind == "multiclass":
        rows.append(
            {"area": "Karabo-Utility", "metric": "f1_retention>=0.80", "value": summary.get("f1_retention_ratio"), "pass": _metric_val(summary, "f1_retention_ratio", 0.0) >= F1_RETENTION_THRESHOLD}
        )
    rows.extend([
        {"area": "Karabo-Privacy", "metric": "synthetic_duplicate_rate", "value": summary.get("synthetic_duplicate_rate"), "pass": True},
        {"area": "Karabo-Privacy", "metric": "replaced_exact_match_rate<0.001", "value": summary.get("replaced_exact_match_rate"), "pass": _metric_val(summary, "replaced_exact_match_rate", 999.0) < 0.001},
    ])
    return pd.DataFrame(rows)


def compute_karabo_metrics(
    df_actual,
    df_out,
    suppressed,
    *,
    feature_categorical_cols,
    numerical_cols,
    tvd_categorical_cols,
    qi_feature_cols,
    qi_cols,
    target_col,
    target_kind=None,
    random_state=42,
):
    category_metrics = categorical_distribution_metrics(df_actual, df_out, tvd_categorical_cols)
    numeric_metric_cols = list(numerical_cols)
    if target_col and target_kind == "numeric" and target_col not in numeric_metric_cols:
        numeric_metric_cols.append(target_col)
    numeric_metrics = numerical_distribution_metrics(df_actual, df_out, numeric_metric_cols)
    rare_category = rare_category_metrics(df_actual, df_out, tvd_categorical_cols)
    relationship_metrics = (
        target_relationship_metrics(df_actual, df_out, qi_feature_cols, target_col, target_kind)
        if target_col and target_kind else pd.DataFrame()
    )
    num_correlation_metrics = numerical_correlation_metrics(df_actual, df_out, numerical_cols)
    cat_pair_metrics = categorical_pair_relationship_metrics(df_actual, df_out, feature_categorical_cols)
    cat_num_metrics = categorical_numerical_relationship_metrics(
        df_actual, df_out, feature_categorical_cols, numerical_cols
    )
    iv_metrics = (
        iv_retention_metrics(df_actual, df_out, qi_feature_cols, target_col, target_kind)
        if target_col and target_kind else pd.DataFrame()
    )
    target_metrics = (
        target_distribution_metrics(df_actual, df_out, target_col, target_kind)
        if target_col and target_kind else pd.DataFrame()
    )
    privacy = privacy_metrics(df_actual, df_out, qi_cols, suppressed)
    utility = (
        utility_metrics(df_actual, df_out, qi_feature_cols, target_col, target_kind, random_state=random_state)
        if target_col and target_kind else pd.DataFrame()
    )

    mean_corr_drift = float(relationship_metrics["drift"].mean()) if len(relationship_metrics) else 0.0
    mean_psi = float(numeric_metrics["psi"].dropna().mean()) if len(numeric_metrics) and "psi" in numeric_metrics else float("nan")
    mean_category_coverage = float(category_metrics["category_coverage"].mean()) if len(category_metrics) else float("nan")
    total_new_categories = int(category_metrics["new_synthetic_category_count"].sum()) if len(category_metrics) else 0
    target_rate_drift = float(target_metrics["rate_drift"].iloc[0]) if len(target_metrics) else float("nan")
    target_ks = float(target_metrics["target_ks"].iloc[0]) if len(target_metrics) and "target_ks" in target_metrics.columns and pd.notna(target_metrics["target_ks"].iloc[0]) else float("nan")
    auc_retention_ratio = float("nan")
    real_auc = float("nan")
    synthetic_auc = float("nan")
    f1_retention_ratio = float("nan")
    real_f1 = float("nan")
    synthetic_f1 = float("nan")
    r2_retention_ratio = float("nan")
    rmse_retention_ratio = float("nan")
    real_r2 = float("nan")
    synthetic_r2 = float("nan")
    real_rmse = float("nan")
    synthetic_rmse = float("nan")
    roc_curves = {}
    if len(utility) and "auc_retention_ratio" in utility.columns:
        vals = utility["auc_retention_ratio"].dropna()
        if len(vals):
            auc_retention_ratio = float(vals.iloc[0])
    if len(utility) and "f1_retention_ratio" in utility.columns:
        vals = utility["f1_retention_ratio"].dropna()
        if len(vals):
            f1_retention_ratio = float(vals.iloc[0])
    if len(utility) and "r2_retention_ratio" in utility.columns:
        vals = utility["r2_retention_ratio"].dropna()
        if len(vals):
            r2_retention_ratio = float(vals.iloc[0])
    if len(utility) and "rmse_retention_ratio" in utility.columns:
        vals = utility["rmse_retention_ratio"].dropna()
        if len(vals):
            rmse_retention_ratio = float(vals.iloc[0])
    if hasattr(utility, "attrs"):
        real_auc = float(utility.attrs.get("real_auc", float("nan")))
        synthetic_auc = float(utility.attrs.get("synthetic_auc", float("nan")))
        real_f1 = float(utility.attrs.get("real_f1", float("nan")))
        synthetic_f1 = float(utility.attrs.get("synthetic_f1", float("nan")))
        real_r2 = float(utility.attrs.get("real_r2", float("nan")))
        synthetic_r2 = float(utility.attrs.get("synthetic_r2", float("nan")))
        real_rmse = float(utility.attrs.get("real_rmse", float("nan")))
        synthetic_rmse = float(utility.attrs.get("synthetic_rmse", float("nan")))
        roc_curves = utility.attrs.get("roc_curves", {})

    karabo_summary = {
        "target_kind": target_kind,
        "mean_category_coverage": round(mean_category_coverage, 4) if pd.notna(mean_category_coverage) else None,
        "total_new_categories": total_new_categories,
        "mean_psi": round(mean_psi, 4) if pd.notna(mean_psi) else None,
        "mean_num_corr_drift": round(float(num_correlation_metrics["abs_corr_diff"].dropna().mean()), 4)
        if len(num_correlation_metrics) else None,
        "mean_cat_pair_drift": round(float(cat_pair_metrics["abs_cramers_v_diff"].dropna().mean()), 4)
        if len(cat_pair_metrics) else None,
        "mean_cat_num_drift": round(float(cat_num_metrics["normalised_group_mean_diff"].dropna().mean()), 4)
        if len(cat_num_metrics) else None,
        "mean_iv_retention": round(float(iv_metrics["retention_ratio"].dropna().mean()), 4)
        if len(iv_metrics) and "retention_ratio" in iv_metrics.columns else None,
        "target_rate_drift": round(target_rate_drift, 4) if pd.notna(target_rate_drift) else None,
        "target_ks": round(target_ks, 4) if pd.notna(target_ks) else None,
        "real_auc": round(real_auc, 4) if pd.notna(real_auc) else None,
        "synthetic_auc": round(synthetic_auc, 4) if pd.notna(synthetic_auc) else None,
        "auc_retention_ratio": round(auc_retention_ratio, 4) if pd.notna(auc_retention_ratio) else None,
        "real_f1": round(real_f1, 4) if pd.notna(real_f1) else None,
        "synthetic_f1": round(synthetic_f1, 4) if pd.notna(synthetic_f1) else None,
        "f1_retention_ratio": round(f1_retention_ratio, 4) if pd.notna(f1_retention_ratio) else None,
        "real_r2": round(real_r2, 4) if pd.notna(real_r2) else None,
        "synthetic_r2": round(synthetic_r2, 4) if pd.notna(synthetic_r2) else None,
        "real_rmse": round(real_rmse, 4) if pd.notna(real_rmse) else None,
        "synthetic_rmse": round(synthetic_rmse, 4) if pd.notna(synthetic_rmse) else None,
        "r2_retention_ratio": round(r2_retention_ratio, 4) if pd.notna(r2_retention_ratio) else None,
        "rmse_retention_ratio": round(rmse_retention_ratio, 4) if pd.notna(rmse_retention_ratio) else None,
        "synthetic_duplicate_rate": float(privacy["synthetic_duplicate_rate"].iloc[0]) if len(privacy) else None,
        "replaced_exact_match_rate": float(privacy["replaced_exact_match_rate"].iloc[0]) if len(privacy) else None,
        "mean_corr_drift": round(mean_corr_drift, 6),
    }
    karabo_scorecard = build_karabo_scorecard(karabo_summary)

    return {
        "category_metrics": category_metrics,
        "numeric_metrics": numeric_metrics,
        "rare_category_metrics": rare_category,
        "relationship_metrics": relationship_metrics,
        "num_correlation_metrics": num_correlation_metrics,
        "cat_pair_relationship_metrics": cat_pair_metrics,
        "cat_num_relationship_metrics": cat_num_metrics,
        "iv_metrics": iv_metrics,
        "target_metrics": target_metrics,
        "privacy_metrics": privacy,
        "utility_metrics": utility,
        "roc_curves": roc_curves,
        "karabo_scorecard": karabo_scorecard,
        "karabo_summary": karabo_summary,
    }


print("validation metrics loaded.")

validation metrics loaded.


## Pipeline functions


In [4]:
TARGET_NAME_HINTS = ("churn", "target", "label", "class", "outcome", "flag", "y")
ID_NAME_HINTS = ("id", "uuid", "key")
HIGH_CARDINALITY_CAT_THRESHOLD = 50


def is_fully_empty(series: pd.Series) -> bool:
    return series.isna().all()


def clean_loaded_dataframe(df: pd.DataFrame, drop_fully_empty: bool = True):
    work = df.copy()
    for col in work.select_dtypes(include=["object", "string"]).columns:
        work[col] = work[col].replace(r"^\s*$", np.nan, regex=True)
    empty_columns = [col for col in work.columns if is_fully_empty(work[col])]
    if drop_fully_empty and empty_columns:
        work = work.drop(columns=empty_columns)
    return work.reset_index(drop=True), empty_columns


def identify_skipped_columns(df: pd.DataFrame, feature_pool, exclude_cols=None, skip_constant_cols=False):
    exclude_cols = set(exclude_cols or [])
    skipped = {}
    for col in feature_pool:
        if col in exclude_cols:
            skipped[col] = "excluded_manual"
        elif skip_constant_cols and df[col].nunique(dropna=True) <= 1:
            skipped[col] = "excluded_constant"
    return skipped


def build_column_validation_table(df, schema, empty_columns=None):
    empty_columns = set(empty_columns or [])
    target_col = schema.get("target_col")
    categorical_cols = set(schema.get("categorical_cols", []))
    numerical_cols = set(schema.get("numerical_cols", []))
    skipped_cols = schema.get("skipped_cols", {})
    passthrough_cols = set(schema.get("passthrough_cols", []))
    id_cols = set(schema.get("id_cols", []))

    rows = []
    for col in df.columns:
        if col in id_cols:
            role = "id"
        elif col == target_col:
            role = "target"
        elif col in categorical_cols:
            role = "categorical_feature"
        elif col in numerical_cols:
            role = "numerical_feature"
        elif col in passthrough_cols:
            role = "passthrough"
        else:
            role = "other"

        non_missing_unique = int(df[col].nunique(dropna=True))
        if col in empty_columns:
            action = "dropped_empty"
        elif col in skipped_cols:
            action = skipped_cols[col]
        else:
            action = "kept"

        rows.append({
            "column": col,
            "role": role,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_rate": round(float(df[col].isna().mean()), 6),
            "unique_count_including_missing": int(df[col].nunique(dropna=False)),
            "unique_count_excluding_missing": non_missing_unique,
            "is_constant": non_missing_unique <= 1,
            "is_high_cardinality": (
                non_missing_unique > HIGH_CARDINALITY_CAT_THRESHOLD
                if role == "categorical_feature" else False
            ),
            "action": action,
        })
    return pd.DataFrame(rows)


def classify_target_kind(series: pd.Series, kind_override=None) -> str | None:
    if kind_override:
        return kind_override
    if series.dtype == object or str(series.dtype) == "category":
        n = series.nunique(dropna=True)
        if n <= 1:
            return None
        return "binary" if n == 2 else "multiclass"
    if pd.api.types.is_numeric_dtype(series):
        nums = pd.to_numeric(series, errors="coerce")
        if nums.notna().sum() < len(series) * 0.5:
            return "multiclass"
        n = nums.nunique(dropna=True)
        if n <= 1:
            return None
        if n == 2:
            return "binary"
        if n <= CAT_TARGET_MAX_UNIQUE:
            return "multiclass"
        return "numeric"
    n = series.nunique(dropna=True)
    if n <= 1:
        return None
    return "binary" if n == 2 else "multiclass"


def _matches_target_hint(col: str) -> bool:
    lc = col.lower().replace("-", "_")
    parts = lc.split("_")
    for hint in TARGET_NAME_HINTS:
        if len(hint) == 1:
            if lc == hint:
                return True
        elif hint in parts or lc == hint:
            return True
    return False


def infer_target_column(df: pd.DataFrame, pool, target_col_override=None):
    target_col = target_col_override
    if not target_col:
        for col in pool:
            if _matches_target_hint(col):
                target_col = col
                break
    if not target_col:
        binary_cols = [c for c in pool if classify_target_kind(df[c]) == "binary"]
        for col in pool:
            if _matches_target_hint(col) and col in binary_cols:
                target_col = col
                break
        if not target_col and binary_cols:
            target_col = binary_cols[-1]
    return target_col


def infer_dataset_schema(
    df: pd.DataFrame,
    target_col_override=None,
    target_kind_override=None,
    id_cols_override=None,
    exclude_cols_override=None,
    skip_constant_cols=False,
):
    id_cols = list(id_cols_override or [])
    if not id_cols:
        for col in df.columns:
            lc = col.lower()
            if df[col].nunique(dropna=True) == len(df) and (
                any(h in lc for h in ID_NAME_HINTS) or lc.endswith("_id") or lc == "id"
            ):
                id_cols.append(col)

    pool = [c for c in df.columns if c not in set(id_cols)]
    target_col = infer_target_column(df, pool, target_col_override)
    target_kind = classify_target_kind(df[target_col], target_kind_override) if target_col else None

    if target_col and target_col in set(exclude_cols_override or []):
        raise ValueError(f"target column cannot be in EXCLUDE_COLS_OVERRIDE: {target_col}")

    feature_pool = [c for c in pool if c != target_col]
    skipped_cols = identify_skipped_columns(
        df, feature_pool, exclude_cols=exclude_cols_override, skip_constant_cols=skip_constant_cols
    )
    active_features = [c for c in feature_pool if c not in skipped_cols]
    passthrough_cols = [c for c in feature_pool if c in skipped_cols]

    categorical_cols, numerical_cols = [], []
    for col in active_features:
        series = df[col]
        if pd.api.types.is_numeric_dtype(series):
            numerical_cols.append(col)
        else:
            categorical_cols.append(col)

    kanon_qi_cols = list(categorical_cols)
    for col in numerical_cols:
        if df[col].nunique(dropna=True) <= 100:
            kanon_qi_cols.append(col)

    generalization = {}
    for col in kanon_qi_cols:
        nums = pd.to_numeric(df[col], errors="coerce")
        if nums.notna().mean() < 0.5:
            continue
        nunique = nums.nunique(dropna=True)
        if nunique <= 1:
            continue
        if nunique > 30:
            generalization[col] = {"type": "quantile_bins", "q": 20}
        elif nunique > 8:
            span = float(nums.max() - nums.min())
            step = max(1, int(round(span / 20)))
            generalization[col] = {"type": "bin_floor", "step": step}

    tvd_categorical_cols = list(categorical_cols)
    if target_col and target_kind in {"binary", "multiclass"} and target_col not in tvd_categorical_cols:
        tvd_categorical_cols.append(target_col)
    qi_feature_cols = categorical_cols + numerical_cols
    qi_cols = qi_feature_cols + ([target_col] if target_col and target_col not in qi_feature_cols else [])

    if target_col and target_col in categorical_cols:
        raise ValueError(f"target column must not be treated as feature categorical: {target_col}")
    missing = [c for c in qi_cols if c not in df.columns]
    if missing:
        raise ValueError(f"inferred QI columns missing from data: {missing}")

    return {
        "id_cols": id_cols,
        "target_col": target_col,
        "target_kind": target_kind,
        "relationship_col": target_col,
        "categorical_cols": categorical_cols,
        "numerical_cols": numerical_cols,
        "kanon_qi_cols": kanon_qi_cols,
        "generalization": generalization,
        "tvd_categorical_cols": tvd_categorical_cols,
        "qi_feature_cols": qi_feature_cols,
        "qi_cols": qi_cols,
        "skipped_cols": skipped_cols,
        "passthrough_cols": passthrough_cols,
    }


def apply_schema(schema: dict):
    global ID_COLS, TARGET_COL, TARGET_KIND, RELATIONSHIP_COL, CATEGORICAL_COLS, NUMERICAL_COLS
    global KANON_QI_COLS, GENERALIZATION, TVD_CATEGORICAL_COLS, QI_FEATURE_COLS, QI_COLS
    global SKIPPED_COLS, PASSTHROUGH_COLS
    ID_COLS = schema["id_cols"]
    TARGET_COL = schema["target_col"]
    TARGET_KIND = schema["target_kind"]
    RELATIONSHIP_COL = schema["relationship_col"]
    CATEGORICAL_COLS = schema["categorical_cols"]
    NUMERICAL_COLS = schema["numerical_cols"]
    KANON_QI_COLS = schema["kanon_qi_cols"]
    GENERALIZATION = schema["generalization"]
    TVD_CATEGORICAL_COLS = schema["tvd_categorical_cols"]
    QI_FEATURE_COLS = schema["qi_feature_cols"]
    QI_COLS = schema["qi_cols"]
    SKIPPED_COLS = schema.get("skipped_cols", {})
    PASSTHROUGH_COLS = schema.get("passthrough_cols", [])


def generalize_for_kanonymity(df, qi_cols):
    g = df[qi_cols].copy()
    for col, rule in GENERALIZATION.items():
        if col not in g.columns:
            continue
        if rule["type"] == "bin_floor":
            s = pd.to_numeric(g[col], errors="coerce")
            step = float(rule["step"])
            g[col] = (s // step) * step
        elif rule["type"] == "quantile_bins":
            ranked = pd.to_numeric(g[col], errors="coerce").rank(method="first")
            g[col] = pd.qcut(ranked, q=int(rule.get("q", 20)), duplicates="drop").astype(str)
    str_cols = g.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    for col in CATEGORICAL_COLS:
        if col in g.columns:
            g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    return g


def identify_suppressed(df, k_anonymity):
    generalized = generalize_for_kanonymity(df, KANON_QI_COLS)
    class_sizes = generalized.groupby(list(generalized.columns), dropna=False).transform("size")
    suppressed = class_sizes < k_anonymity
    pool_idx = np.where(~suppressed.values)[0]
    synth_idx = np.where(suppressed.values)[0]
    if len(pool_idx) == 0:
        raise ValueError(f"k_anonymity={k_anonymity}: no neighbour pool")
    return suppressed, pool_idx, synth_idx


def fit_preprocessors(df, scaler_method):
    cat_encoders = {}
    for col in CATEGORICAL_COLS:
        enc = LabelEncoder()
        enc.fit(df[col].astype(str).fillna(MISSING_LABEL))
        cat_encoders[col] = enc
    num_medians = df[NUMERICAL_COLS].median()
    num_filled = df[NUMERICAL_COLS].fillna(num_medians).values.astype(float)
    scaler_cls = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}[scaler_method]
    num_scaler = scaler_cls().fit(num_filled)
    num_p01 = {c: np.percentile(df[c].dropna(), 1) for c in NUMERICAL_COLS}
    num_p99 = {c: np.percentile(df[c].dropna(), 99) for c in NUMERICAL_COLS}
    X_cat = np.column_stack([
        cat_encoders[c].transform(df[c].astype(str).fillna(MISSING_LABEL)) for c in CATEGORICAL_COLS
    ]) if CATEGORICAL_COLS else np.empty((len(df), 0))
    X_num = num_scaler.transform(num_filled)
    num_ranges = np.ptp(num_filled, axis=0)
    target_bounds = {}
    if TARGET_COL and TARGET_KIND == "numeric" and TARGET_COL in df.columns:
        target_vals = pd.to_numeric(df[TARGET_COL], errors="coerce").dropna()
        if len(target_vals):
            target_bounds = {
                "p01": float(np.percentile(target_vals, 1)),
                "p99": float(np.percentile(target_vals, 99)),
            }
    return {
        "cat_encoders": cat_encoders,
        "num_medians": num_medians,
        "num_scaler": num_scaler,
        "num_p01": num_p01,
        "num_p99": num_p99,
        "X_cat": X_cat,
        "X_num": X_num,
        "num_filled": num_filled,
        "num_ranges": num_ranges,
        "target_bounds": target_bounds,
    }


def categorical_pairwise_distance(pool_cat, synth_cat, metric):
    mismatch = pool_cat[np.newaxis, :, :] != synth_cat[:, np.newaxis, :]
    n_cat = pool_cat.shape[1]
    if n_cat == 0:
        return np.zeros((len(synth_cat), len(pool_cat)))
    matches = (~mismatch).sum(axis=2).astype(float)
    if metric == "hamming":
        return np.mean(mismatch, axis=2)
    if metric == "jaccard":
        union = 2.0 * n_cat - matches
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(union > 0, matches / union, 1.0)
        return 1.0 - sim
    if metric == "overlap":
        return 1.0 - matches / n_cat
    if metric == "cosine":
        pool = pool_cat.astype(float)
        synth = synth_cat.astype(float)
        dot = synth @ pool.T
        pool_norm = np.linalg.norm(pool, axis=1)
        synth_norm = np.linalg.norm(synth, axis=1)
        denom = synth_norm[:, np.newaxis] * pool_norm[np.newaxis, :]
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(denom > 0, dot / denom, 1.0)
        return 1.0 - sim
    raise ValueError(f"Unknown cat distance: {metric!r}; use one of {CAT_DISTANCE_METRICS}")


def build_neighbor_cache(cfg, prep, pool_idx, synth_idx):
    pool_cat = prep["X_cat"][pool_idx]
    synth_cat = prep["X_cat"][synth_idx]
    if cfg["distance_mode"] == "gower":
        pool_num = prep["num_filled"][pool_idx]
        synth_num = prep["num_filled"][synth_idx]
        n_synth, n_pool = len(synth_num), len(pool_num)
        total_dist = np.zeros((n_synth, n_pool))
        safe_ranges = np.where(prep["num_ranges"] < 1e-8, 1.0, prep["num_ranges"])
        for j in range(pool_num.shape[1]):
            total_dist += np.abs(pool_num[np.newaxis, :, j] - synth_num[:, np.newaxis, j]) / safe_ranges[j]
        for j in range(pool_cat.shape[1]):
            total_dist += (pool_cat[np.newaxis, :, j] != synth_cat[:, np.newaxis, j]).astype(float)
        total_dist /= max(1, pool_num.shape[1] + pool_cat.shape[1])
    else:
        pool_num = prep["X_num"][pool_idx]
        synth_num = prep["X_num"][synth_idx]
        diff = pool_num[np.newaxis, :, :] - synth_num[:, np.newaxis, :]
        metric = cfg["num_distance_metric"]
        if metric == "euclidean":
            num_dist = np.linalg.norm(diff, axis=2)
        elif metric == "manhattan":
            num_dist = np.sum(np.abs(diff), axis=2)
        elif metric == "minkowski":
            p = int(cfg["minkowski_p"])
            num_dist = np.sum(np.abs(diff) ** p, axis=2) ** (1.0 / p)
        else:
            raise ValueError(metric)
        cat_dist = categorical_pairwise_distance(pool_cat, synth_cat, cfg["cat_distance_metric"])
        total_dist = float(cfg["num_weight"]) * num_dist + float(cfg["cat_weight"]) * cat_dist

    k_cache = min(int(cfg["k_neighbors"]), len(pool_idx))
    cache = {}
    for local_i, global_i in enumerate(synth_idx):
        row_dist = total_dist[local_i]
        idx_local = np.argpartition(row_dist, k_cache - 1)[:k_cache]
        order = idx_local[np.argsort(row_dist[idx_local])]
        cache[int(global_i)] = (row_dist[order], pool_idx[order])
    return cache


def generate_categorical(vals, weights, method, rng):
    if method == "mode":
        return Counter(vals).most_common(1)[0][0]
    if method == "weighted_mode":
        counts = defaultdict(float)
        for v, w in zip(vals, weights):
            counts[str(v)] += w
        return max(counts, key=counts.get)
    if method == "probability":
        return str(rng.choice(vals, p=weights / weights.sum()))
    raise ValueError(method)


def _normalize_target_gen_config(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    s = str(raw).strip()
    if s in ("", "auto", "none", "None"):
        return None
    return s


def effective_target_gen_method(row, target_kind):
    """Resolve target synthesis method from a parameter row (no globals)."""
    if target_kind == "numeric":
        method = str(row["num_gen_method"])
        if method not in NUMERIC_TARGET_GEN_METHODS:
            raise ValueError(
                f"num_gen_method={method!r} invalid for numeric target; "
                f"use one of {NUMERIC_TARGET_GEN_METHODS}"
            )
        return method
    cfg = _normalize_target_gen_config(row.get("target_gen_method"))
    if cfg is None:
        return str(row["cat_gen_method"])
    if cfg not in DISCRETE_TARGET_GEN_METHODS:
        raise ValueError(
            f"target_gen_method={cfg!r} invalid for {target_kind} target; "
            f"use one of {DISCRETE_TARGET_GEN_METHODS} or None to follow cat_gen_method"
        )
    return cfg


def resolve_target_gen_method(cfg):
    """Effective target synthesis method for the current TARGET_KIND."""
    if not TARGET_COL:
        return None
    return effective_target_gen_method(cfg, TARGET_KIND)


def _row_is_valid_config(row):
    cfg = {
        "distance_mode": str(row["distance_mode"]),
        "cat_distance_metric": str(row["cat_distance_metric"]),
        "scaler_method": str(row["scaler_method"]),
        "num_distance_metric": str(row["num_distance_metric"]),
        "minkowski_p": int(row["minkowski_p"]),
        "num_weight": float(row["num_weight"]),
        "cat_weight": float(row["cat_weight"]),
    }
    return is_valid_config(cfg)


def _dedupe_gower_grid(combos):
    """Gower ignores cat/num metric grid dims — collapse duplicate rows."""
    if "distance_mode" not in combos.columns or not (combos["distance_mode"] == "gower").any():
        return combos
    gower = combos[combos["distance_mode"] == "gower"].copy()
    non_gower = combos[combos["distance_mode"] != "gower"]
    ignore_cols = {"cat_distance_metric", "num_distance_metric", "minkowski_p"}
    dedup_cols = [c for c in gower.columns if c not in ignore_cols]
    gower = gower.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
    gower["cat_distance_metric"] = "hamming"
    gower["num_distance_metric"] = "euclidean"
    gower["minkowski_p"] = 3
    return pd.concat([non_gower, gower], ignore_index=True)


def finalize_parameter_combinations(combos, target_kind):
    """Drop redundant grid rows and normalize target_gen_method per target kind."""
    combos = combos.copy()
    if target_kind == "numeric":
        combos = combos.drop(columns=["target_gen_method"]).drop_duplicates().reset_index(drop=True)
        combos["target_gen_method"] = None
    else:
        combos["target_gen_method"] = combos["target_gen_method"].map(_normalize_target_gen_config)
        combos["_effective_target_gen"] = combos.apply(
            lambda row: effective_target_gen_method(row, target_kind), axis=1
        )
        id_cols = [c for c in combos.columns if c not in ("target_gen_method", "_effective_target_gen")]
        combos = combos.drop_duplicates(subset=id_cols + ["_effective_target_gen"]).reset_index(drop=True)

        def _collapse_target_config(row):
            cfg = row["target_gen_method"]
            eff = row["_effective_target_gen"]
            cat = str(row["cat_gen_method"])
            if cfg is not None and str(cfg) == eff:
                return str(cfg) if str(cfg) != cat else None
            if cfg is None and eff == cat:
                return None
            return cfg

        combos["target_gen_method"] = combos.apply(_collapse_target_config, axis=1)
        combos = combos.drop(columns=["_effective_target_gen"])

    combos = _dedupe_gower_grid(combos)
    combos = combos[combos.apply(_row_is_valid_config, axis=1)].reset_index(drop=True)
    return combos


def _synthesize_discrete_target(df, nbrs, w, cfg, rng):
    vals = df.loc[nbrs, TARGET_COL].astype(str).fillna(MISSING_LABEL).values
    return generate_categorical(vals, w, resolve_target_gen_method(cfg), rng)


def _cast_column_value(col, val, df):
    is_discrete_target = TARGET_COL and col == TARGET_COL and TARGET_KIND in {"binary", "multiclass"}
    is_numeric_target = TARGET_COL and col == TARGET_COL and TARGET_KIND == "numeric"
    cat_cols = set(CATEGORICAL_COLS) | ({TARGET_COL} if is_discrete_target else set())
    if col in cat_cols and col in df.columns and pd.api.types.is_integer_dtype(df[col]):
        return int(float(val)) if str(val).replace(".", "", 1).isdigit() else val
    if (col in NUMERICAL_COLS or is_numeric_target) and col in df.columns:
        if is_numeric_target or pd.api.types.is_integer_dtype(df[col]):
            return int(round(val)) if pd.api.types.is_integer_dtype(df[col]) else float(val)
    return val


def _synthesize_numeric_target(df, row_idx, nbrs, w, cfg, rng):
    method = resolve_target_gen_method(cfg)
    neighbor_vals = pd.to_numeric(df.loc[nbrs, TARGET_COL], errors="coerce").values.astype(float)
    orig = float(pd.to_numeric(df.loc[row_idx, TARGET_COL], errors="coerce"))
    if method == "interpolation":
        nj = int(rng.choice(nbrs))
        nval = float(pd.to_numeric(df.loc[nj, TARGET_COL], errors="coerce"))
        t = float(rng.random())
        return orig + t * (nval - orig)
    if method == "weighted_mean":
        return float(np.dot(w, neighbor_vals))
    raise ValueError(method)


def synthesize_dataset(df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache):
    rng = np.random.default_rng(int(cfg.get("random_state", RANDOM_STATE)))
    df_out = df.copy()
    X_num = prep["X_num"]
    synth_cat_cols = list(CATEGORICAL_COLS)

    for row_idx in synth_idx:
        row_idx = int(row_idx)
        dists_full, nbrs_full = neighbor_cache[row_idx]
        k = min(int(cfg["k_neighbors"]), len(dists_full))
        dists, nbrs = dists_full[:k], nbrs_full[:k]
        w = 1.0 / (dists + 1e-8)
        w = w / w.sum()
        row_out = {}
        synthesis_mode = str(cfg.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)).lower()

        if synthesis_mode == "donor":
            neighbour_pool = df.loc[nbrs]
            for col in synth_cat_cols:
                vals = neighbour_pool[col].astype(str).fillna(MISSING_LABEL).values
                row_out[col] = str(rng.choice(vals, p=w))
            synth_scaled = np.zeros(len(NUMERICAL_COLS))
            donor = int(rng.choice(nbrs, p=w))
            for j in range(len(NUMERICAL_COLS)):
                t = float(rng.uniform(0.1, 0.9))
                synth_scaled[j] = X_num[row_idx, j] + t * (X_num[donor, j] - X_num[row_idx, j])
        else:
            for col in synth_cat_cols:
                vals = df.loc[nbrs, col].astype(str).fillna(MISSING_LABEL).values
                row_out[col] = generate_categorical(vals, w, cfg["cat_gen_method"], rng)
            synth_scaled = np.zeros(len(NUMERICAL_COLS))
            for j in range(len(NUMERICAL_COLS)):
                if cfg["num_gen_method"] == "interpolation":
                    nj = rng.choice(nbrs)
                    t = rng.random()
                    synth_scaled[j] = X_num[row_idx, j] + t * (X_num[nj, j] - X_num[row_idx, j])
                elif cfg["num_gen_method"] == "weighted_mean":
                    synth_scaled[j] = float(np.dot(w, X_num[nbrs, j]))
                else:
                    raise ValueError(cfg["num_gen_method"])

        if TARGET_COL and TARGET_COL in df.columns:
            if TARGET_KIND in {"binary", "multiclass"}:
                row_out[TARGET_COL] = _synthesize_discrete_target(df, nbrs, w, cfg, rng)
            elif TARGET_KIND == "numeric":
                synth_target = _synthesize_numeric_target(df, row_idx, nbrs, w, cfg, rng)
                bounds = prep.get("target_bounds", {})
                if bounds:
                    synth_target = float(np.clip(synth_target, bounds["p01"], bounds["p99"]))
                row_out[TARGET_COL] = synth_target

        synth_num = prep["num_scaler"].inverse_transform(synth_scaled.reshape(1, -1)).flatten()
        for j, col in enumerate(NUMERICAL_COLS):
            row_out[col] = float(np.clip(synth_num[j], prep["num_p01"][col], prep["num_p99"][col]))

        for col in QI_COLS:
            df_out.loc[row_idx, col] = _cast_column_value(col, row_out[col], df)
    return df_out


def compute_metrics(df_actual, df_out, suppressed, relationship_col=None):
    target_col = relationship_col or TARGET_COL
    karabo = compute_karabo_metrics(
        df_actual, df_out, suppressed,
        feature_categorical_cols=CATEGORICAL_COLS,
        numerical_cols=NUMERICAL_COLS,
        tvd_categorical_cols=TVD_CATEGORICAL_COLS,
        qi_feature_cols=QI_FEATURE_COLS,
        qi_cols=QI_COLS,
        target_col=target_col,
        target_kind=TARGET_KIND,
        random_state=RANDOM_STATE,
    )
    category_metrics = karabo["category_metrics"]
    numeric_metrics = karabo["numeric_metrics"]
    tvd_pass_rate = float(category_metrics["pass_tvd"].mean())
    ks_pass_rate = float(numeric_metrics["pass_ks"].mean())
    mean_corr_drift = float(karabo["karabo_summary"]["mean_corr_drift"])
    exact_match_rate = float(karabo["karabo_summary"]["replaced_exact_match_rate"])
    summary = karabo["karabo_summary"]

    scorecard_rows = [
        {"area": "Quality-Categorical", "metric": "tvd_pass_rate>=0.85", "value": round(tvd_pass_rate, 4), "pass": tvd_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "ks_pass_rate>=0.85", "value": round(ks_pass_rate, 4), "pass": ks_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "mean_KS<0.10", "value": round(numeric_metrics["KS_statistic"].mean(), 4), "pass": numeric_metrics["KS_statistic"].mean() < KS_THRESHOLD},
        {"area": "Relationships", "metric": "mean_corr_drift<0.05", "value": round(mean_corr_drift, 4), "pass": mean_corr_drift < 0.05},
        {"area": "Privacy", "metric": "replaced_exact_match_rate<0.001", "value": round(exact_match_rate, 6), "pass": exact_match_rate < 0.001},
        {"area": "Suppression", "metric": "recovery_rate", "value": 1.0, "pass": suppressed.sum() > 0},
    ]
    if TARGET_KIND == "numeric":
        scorecard_rows.extend([
            {"area": "Target-Numeric", "metric": "target_rate_drift<0.05", "value": summary.get("target_rate_drift"), "pass": _metric_val(summary, "target_rate_drift", 999.0) < TARGET_RATE_DRIFT_THRESHOLD},
            {"area": "Target-Numeric", "metric": "target_ks<0.10", "value": summary.get("target_ks"), "pass": _metric_val(summary, "target_ks", 999.0) < TARGET_KS_THRESHOLD},
            {"area": "Utility-Regression", "metric": "r2_retention>=0.80", "value": summary.get("r2_retention_ratio"), "pass": _metric_val(summary, "r2_retention_ratio", 0.0) >= R2_RETENTION_THRESHOLD},
            {"area": "Utility-Regression", "metric": "rmse_retention>=0.80", "value": summary.get("rmse_retention_ratio"), "pass": _metric_val(summary, "rmse_retention_ratio", 0.0) >= RMSE_RETENTION_THRESHOLD},
        ])
    elif TARGET_KIND == "binary":
        scorecard_rows.extend([
            {"area": "Target-Binary", "metric": "target_rate_drift<0.05", "value": summary.get("target_rate_drift"), "pass": _metric_val(summary, "target_rate_drift", 999.0) < TARGET_RATE_DRIFT_THRESHOLD},
            {"area": "Utility-Classification", "metric": "auc_retention>=0.80", "value": summary.get("auc_retention_ratio"), "pass": _metric_val(summary, "auc_retention_ratio", 0.0) >= AUC_RETENTION_THRESHOLD},
            {"area": "Utility-Classification", "metric": "f1_retention>=0.80", "value": summary.get("f1_retention_ratio"), "pass": _metric_val(summary, "f1_retention_ratio", 0.0) >= F1_RETENTION_THRESHOLD},
        ])
    elif TARGET_KIND == "multiclass":
        scorecard_rows.extend([
            {"area": "Target-Multiclass", "metric": "target_tvd<0.05", "value": summary.get("target_rate_drift"), "pass": _metric_val(summary, "target_rate_drift", 999.0) < TARGET_RATE_DRIFT_THRESHOLD},
            {"area": "Utility-Classification", "metric": "f1_retention>=0.80", "value": summary.get("f1_retention_ratio"), "pass": _metric_val(summary, "f1_retention_ratio", 0.0) >= F1_RETENTION_THRESHOLD},
        ])
    scorecard = pd.DataFrame(scorecard_rows)
    overall_pass = bool(scorecard["pass"].all())
    distribution_summary = build_distribution_summary(df_actual, df_out)

    return {
        **karabo,
        "scorecard": scorecard,
        "distribution_summary": distribution_summary,
        "overall_pass": overall_pass,
        "tvd_pass_rate": round(tvd_pass_rate, 4),
        "ks_pass_rate": round(ks_pass_rate, 4),
        "mean_tvd": round(category_metrics["TVD"].mean(), 4),
        "mean_ks": round(numeric_metrics["KS_statistic"].mean(), 4),
        "mean_corr_drift": round(mean_corr_drift, 6),
        "exact_match_rate": round(exact_match_rate, 6),
        "relationship_col": target_col,
    }


def is_valid_config(cfg):
    if cfg["distance_mode"] == "gower":
        # Gower uses its own mixed-type formula; cat/num metrics are placeholders
        if float(cfg["num_weight"]) != 1.0 or float(cfg["cat_weight"]) != 1.0:
            return False
        return True
    if cfg["num_distance_metric"] != "minkowski" and int(cfg["minkowski_p"]) != 3:
        return False
    return True


def config_to_key(cfg):
    k = int(cfg["k_anonymity"])
    mode = cfg["distance_mode"]
    if mode == "gower":
        return (k, mode)
    return (
        k, cfg["scaler_method"], mode,
        cfg["num_distance_metric"], cfg["cat_distance_metric"], int(cfg["minkowski_p"]),
        float(cfg["num_weight"]), float(cfg["cat_weight"]),
    )


def make_folder_name(idx, cfg):
    cat_tag = f"-cat-{cfg['cat_distance_metric']}" if cfg["distance_mode"] == "weighted_sum" else ""
    tgt_method = resolve_target_gen_method(cfg)
    return (
        f"{idx:03d}_k{int(cfg['k_anonymity'])}_kn{int(cfg['k_neighbors'])}_cat-{cfg['cat_gen_method']}_"
        f"num-{cfg['num_gen_method']}_tgt-{tgt_method}_scale-{cfg['scaler_method']}_{cfg['distance_mode']}-"
        f"{cfg['num_distance_metric']}{cat_tag}_w-{cfg['distance_profile']}"
    )


def make_display_name(cfg):
    cat_m = cfg["cat_distance_metric"] if cfg["distance_mode"] == "weighted_sum" else "gower"
    tgt_method = resolve_target_gen_method(cfg)
    return (
        f"k={int(cfg['k_anonymity'])} kn={int(cfg['k_neighbors'])} "
        f"cat={cfg['cat_gen_method']} num={cfg['num_gen_method']} tgt={tgt_method} "
        f"scale={cfg['scaler_method']} {cfg['distance_mode']}/{cfg['num_distance_metric']}/{cat_m}/{cfg['distance_profile']}"
    )


def finalize_ranking(rows):
    ranking = pd.DataFrame(rows)
    if ranking.empty:
        ranking["rank"] = []
        return ranking

    relationship_score = 1.0 - (ranking["mean_corr_drift"] / 0.05).clip(lower=0.0, upper=1.0)
    ks_quality = 1.0 - (ranking["mean_ks"] / 0.10).clip(lower=0.0, upper=1.0)
    ranking["relationship_score"] = relationship_score.round(4)
    ranking["ks_quality"] = ks_quality.round(4)

    def _utility_score(row):
        kind = row.get("target_kind")
        if kind == "numeric":
            r2 = row.get("r2_retention_ratio")
            rmse = row.get("rmse_retention_ratio")
            parts = [v for v in (r2, rmse) if pd.notna(v)]
            return float(np.mean(parts)) if parts else float("nan")
        if kind == "binary":
            return row.get("auc_retention_ratio")
        return row.get("f1_retention_ratio")

    ranking["utility_score"] = ranking.apply(_utility_score, axis=1)
    utility_score = ranking["utility_score"].fillna(0.0).clip(lower=0.0, upper=1.0)
    ranking["quality_score"] = (
        0.28 * ranking["tvd_pass_rate"]
        + 0.28 * ranking["ks_pass_rate"]
        + 0.18 * relationship_score
        + 0.12 * ks_quality
        + 0.14 * utility_score
    ).round(4)
    runtime_norm = ranking["runtime_sec"] / max(float(ranking["runtime_sec"].max()), 1e-6)
    ranking["composite_score"] = (ranking["quality_score"] - 0.02 * runtime_norm).round(4)
    ranking = ranking.sort_values(
        by=["overall_pass", "composite_score", "mean_corr_drift", "runtime_sec"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    ranking["rank"] = ranking.index + 1
    return ranking


def config_signature_from_cfg(cfg):
    """Stable identity for a run (used to resume / skip duplicates)."""
    return (
        str(RELATIONSHIP_COL or ""),
        str(TARGET_KIND or ""),
        int(cfg["k_anonymity"]),
        int(cfg["k_neighbors"]),
        str(cfg["cat_gen_method"]),
        str(cfg["num_gen_method"]),
        _normalize_target_gen_config(cfg.get("target_gen_method")),
        str(resolve_target_gen_method(cfg)),
        str(cfg["scaler_method"]),
        str(cfg["distance_mode"]),
        str(cfg["num_distance_metric"]),
        str(cfg["cat_distance_metric"]),
        int(cfg["minkowski_p"]),
        float(cfg["num_weight"]),
        float(cfg["cat_weight"]),
        str(cfg["distance_profile"]),
        int(cfg.get("random_state", RANDOM_STATE)),
    )


def config_signature_from_ranking_row(row):
    if not isinstance(row, dict):
        row = row.to_dict()
    raw_tgt = row.get("target_gen_method_config")
    if raw_tgt is None or (isinstance(raw_tgt, float) and pd.isna(raw_tgt)):
        raw_tgt = None
    else:
        raw_tgt = _normalize_target_gen_config(raw_tgt)
    return (
        str(row.get("relationship_col", "") or ""),
        str(row.get("target_kind", "") or ""),
        int(row["k_anonymity"]),
        int(row["k_neighbors"]),
        str(row["cat_gen_method"]),
        str(row["num_gen_method"]),
        raw_tgt,
        str(row["target_gen_method"]),
        str(row["scaler_method"]),
        str(row["distance_mode"]),
        str(row["num_distance_metric"]),
        str(row["cat_distance_metric"]),
        int(row["minkowski_p"]),
        float(row["num_weight"]),
        float(row["cat_weight"]),
        str(row["distance_profile"]),
        int(row.get("random_state", RANDOM_STATE)),
    )


def load_existing_ranking(csv_path=EXPERIMENT_RANKING_CSV):
    path = Path(csv_path)
    if not path.exists():
        return [], set()
    existing = pd.read_csv(path)
    if existing.empty:
        return [], set()
    rows = existing.to_dict("records")
    signatures = {config_signature_from_ranking_row(r) for r in rows}
    return rows, signatures


def row_to_config(row):
    cfg = {
        "k_anonymity": int(row["k_anonymity"]),
        "k_neighbors": int(row["k_neighbors"]),
        "cat_gen_method": str(row["cat_gen_method"]),
        "num_gen_method": str(row["num_gen_method"]),
        "target_gen_method": _normalize_target_gen_config(row.get("target_gen_method")),
        "scaler_method": str(row["scaler_method"]),
        "distance_mode": str(row["distance_mode"]),
        "num_distance_metric": str(row["num_distance_metric"]),
        "minkowski_p": int(row["minkowski_p"]),
        "cat_distance_metric": str(row["cat_distance_metric"]),
        "num_weight": float(row["num_weight"]),
        "cat_weight": float(row["cat_weight"]),
        "distance_profile": str(row["distance_profile"]),
        "random_state": RANDOM_STATE,
        "row_synthesis_mode": str(row.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)),
    }
    return cfg


def build_distribution_summary(df_actual, df_out):
    dist_rows = []
    for col in NUMERICAL_COLS:
        actual = pd.to_numeric(df_actual[col], errors="coerce")
        synth = pd.to_numeric(df_out[col], errors="coerce")
        actual_mean = float(actual.mean())
        synth_mean = float(synth.mean())
        dist_rows.append({
            "column": col,
            "type": "numerical",
            "actual_mean": round(actual_mean, 4),
            "synthetic_mean": round(synth_mean, 4),
            "actual_std": round(float(actual.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "mean_diff_pct": round(abs(synth_mean - actual_mean) / max(abs(actual_mean), 1e-6) * 100, 2),
        })
    if TARGET_COL and TARGET_KIND == "numeric" and TARGET_COL not in NUMERICAL_COLS:
        actual = pd.to_numeric(df_actual[TARGET_COL], errors="coerce")
        synth = pd.to_numeric(df_out[TARGET_COL], errors="coerce")
        actual_mean = float(actual.mean())
        synth_mean = float(synth.mean())
        dist_rows.append({
            "column": TARGET_COL,
            "type": "target_numeric",
            "actual_mean": round(actual_mean, 4),
            "synthetic_mean": round(synth_mean, 4),
            "actual_std": round(float(actual.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "mean_diff_pct": round(abs(synth_mean - actual_mean) / max(abs(actual_mean), 1e-6) * 100, 2),
        })
    for col in TVD_CATEGORICAL_COLS:
        if col not in df_actual.columns:
            continue
        actual_counts = df_actual[col].astype(str).value_counts(normalize=True)
        synth_counts = df_out[col].astype(str).value_counts(normalize=True)
        dist_rows.append({
            "column": col,
            "type": "categorical",
            "actual_category_count": int(actual_counts.shape[0]),
            "synthetic_category_count": int(synth_counts.shape[0]),
            "actual_top_category_pct": round(float(actual_counts.iloc[0]) * 100, 2),
            "synthetic_top_category_pct": round(float(synth_counts.iloc[0]) * 100, 2),
            "mean_diff_pct": np.nan,
        })
    return pd.DataFrame(dist_rows)


def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    return str(obj)


def build_parameters_payload(cfg):
    return {
        **cfg,
        "target_gen_method_effective": resolve_target_gen_method(cfg),
        "data_file": DATA_FILE,
        "target_col": TARGET_COL,
        "target_kind": TARGET_KIND,
        "relationship_col": RELATIONSHIP_COL,
        "id_cols": ID_COLS,
        "categorical_cols": CATEGORICAL_COLS,
        "numerical_cols": NUMERICAL_COLS,
        "kanon_qi_cols": KANON_QI_COLS,
        "qi_cols": QI_COLS,
        "tvd_categorical_cols": TVD_CATEGORICAL_COLS,
        "row_synthesis_mode": ROW_SYNTHESIS_MODE,
        "generalization": GENERALIZATION,
        "skipped_cols": SKIPPED_COLS,
        "passthrough_cols": PASSTHROUGH_COLS,
    }


def save_experiment_outputs(folder_name, df_out, cfg, metrics, *, runtime_sec, peak_memory_mb, n_suppressed):
    out_dir = Path(EXPERIMENT_OUTPUT_DIR) / folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    df_out.to_csv(out_dir / "anonymized_dataset.csv", index=False)

    with open(out_dir / "parameters.json", "w", encoding="utf-8") as f:
        json.dump(build_parameters_payload(cfg), f, indent=2, default=_json_default)

    summary = {
        **metrics.get("karabo_summary", {}),
        "folder": folder_name,
        "name": make_display_name(cfg),
        "runtime_sec": runtime_sec,
        "peak_memory_mb": peak_memory_mb,
        "n_suppressed": n_suppressed,
        "overall_pass": metrics.get("overall_pass"),
        "tvd_pass_rate": metrics.get("tvd_pass_rate"),
        "ks_pass_rate": metrics.get("ks_pass_rate"),
        "mean_tvd": metrics.get("mean_tvd"),
        "mean_ks": metrics.get("mean_ks"),
        "mean_corr_drift": metrics.get("mean_corr_drift"),
        "exact_match_rate": metrics.get("exact_match_rate"),
        "relationship_col": metrics.get("relationship_col"),
    }
    with open(out_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=_json_default)

    metric_tables = {
        "category_metrics": metrics.get("category_metrics"),
        "numeric_metrics": metrics.get("numeric_metrics"),
        "rare_category_metrics": metrics.get("rare_category_metrics"),
        "relationship_metrics": metrics.get("relationship_metrics"),
        "num_correlation_metrics": metrics.get("num_correlation_metrics"),
        "cat_pair_relationship_metrics": metrics.get("cat_pair_relationship_metrics"),
        "cat_num_relationship_metrics": metrics.get("cat_num_relationship_metrics"),
        "iv_metrics": metrics.get("iv_metrics"),
        "target_metrics": metrics.get("target_metrics"),
        "privacy_metrics": metrics.get("privacy_metrics"),
        "utility_metrics": metrics.get("utility_metrics"),
        "scorecard": metrics.get("scorecard"),
        "karabo_scorecard": metrics.get("karabo_scorecard"),
        "distribution_summary": metrics.get("distribution_summary"),
    }
    for name, table in metric_tables.items():
        if isinstance(table, pd.DataFrame) and len(table):
            table.to_csv(out_dir / f"{name}.csv", index=False)

    roc_curves = metrics.get("roc_curves")
    if roc_curves:
        with open(out_dir / "roc_curves.json", "w", encoding="utf-8") as f:
            json.dump(roc_curves, f, indent=2, default=_json_default)

    return out_dir




## Load data


In [5]:
try:
    display
except NameError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string())
        else:
            print(obj)

df = pd.read_csv(DATA_FILE)
df, EMPTY_COLUMNS = clean_loaded_dataframe(df, drop_fully_empty=DROP_FULLY_EMPTY_COLS)

schema = infer_dataset_schema(
    df,
    TARGET_COL_OVERRIDE,
    TARGET_KIND_OVERRIDE,
    ID_COLS_OVERRIDE,
    EXCLUDE_COLS_OVERRIDE,
    SKIP_CONSTANT_COLS,
)
apply_schema(schema)
COLUMN_VALIDATION = build_column_validation_table(df, schema, EMPTY_COLUMNS)
PARAMETER_COMBINATIONS = finalize_parameter_combinations(PARAMETER_COMBINATIONS, TARGET_KIND)

print("Loaded", f"{len(df):,}", "rows from", DATA_FILE)
print("Dropped fully empty columns:", len(EMPTY_COLUMNS))
if EMPTY_COLUMNS:
    print(" ", EMPTY_COLUMNS)
print("ID columns (excluded):", ID_COLS)
print("Skipped / passthrough columns:", PASSTHROUGH_COLS)
if SKIPPED_COLS:
    print(" Skip reasons:", SKIPPED_COLS)
print("Categorical QI (features):", CATEGORICAL_COLS)
print("Target column:", TARGET_COL)
print("Target kind:", TARGET_KIND)
if TARGET_KIND == "numeric":
    print("Target gen: uses num_gen_method", list(NUMERIC_TARGET_GEN_METHODS))
else:
    print("Target gen (discrete):", list(DISCRETE_TARGET_GEN_METHODS), "| None follows cat_gen_method")
print("Grid (after target-kind adjustment):", len(PARAMETER_COMBINATIONS), "configs")
PARAMETER_COMBINATIONS.head()
print("Relationship column:", RELATIONSHIP_COL)
print("Numerical QI:", NUMERICAL_COLS)
print("TVD categorical cols:", TVD_CATEGORICAL_COLS)
print("QI feature cols:", QI_FEATURE_COLS)
print("QI cols (features + target):", QI_COLS)
print("Row synthesis mode:", ROW_SYNTHESIS_MODE)
print("k-anonymity columns:", KANON_QI_COLS)
print("Generalization rules:", GENERALIZATION)
display(COLUMN_VALIDATION)
display(
    COLUMN_VALIDATION.groupby(["role", "action"], dropna=False)
    .size()
    .reset_index(name="column_count")
)
pd.DataFrame({
    "role": ["id", "categorical", "numerical", "target", "relationship", "tvd_categorical", "qi_features", "k-anon", "passthrough"],
    "columns": [
        ID_COLS,
        CATEGORICAL_COLS,
        NUMERICAL_COLS,
        [TARGET_COL] if TARGET_COL else [],
        [RELATIONSHIP_COL] if RELATIONSHIP_COL else [],
        TVD_CATEGORICAL_COLS,
        QI_FEATURE_COLS,
        KANON_QI_COLS,
        PASSTHROUGH_COLS,
    ],
})


Loaded 10,000 rows from ../Bank Customer Churn Prediction.csv
Dropped fully empty columns: 0
ID columns (excluded): ['customer_id']
Skipped / passthrough columns: []
Categorical QI (features): ['country', 'gender']
Target column: estimated_salary
Target kind: numeric
Target gen: uses num_gen_method ['interpolation', 'weighted_mean']
Grid (after target-kind adjustment): 48 configs
Relationship column: estimated_salary
Numerical QI: ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'churn']
TVD categorical cols: ['country', 'gender']
QI feature cols: ['country', 'gender', 'credit_score', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'churn']
QI cols (features + target): ['country', 'gender', 'credit_score', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'churn', 'estimated_salary']
Row synthesis mode: donor
k-anonymity columns: ['country', 'gender', 'age', 'tenure', 'products_nu

,column,role,dtype,missing_count,missing_rate,unique_count_including_missing,unique_count_excluding_missing,is_constant,is_high_cardinality,action
0,customer_id,id,int64,0,0.0,10000,10000,False,False,kept
1,credit_score,numerical_feature,int64,0,0.0,460,460,False,False,kept
2,country,categorical_feature,str,0,0.0,3,3,False,False,kept
3,gender,categorical_feature,str,0,0.0,2,2,False,False,kept
4,age,numerical_feature,int64,0,0.0,70,70,False,False,kept
5,tenure,numerical_feature,int64,0,0.0,11,11,False,False,kept
6,balance,numerical_feature,float64,0,0.0,6382,6382,False,False,kept
7,products_number,numerical_feature,int64,0,0.0,4,4,False,False,kept
8,credit_card,numerical_feature,int64,0,0.0,2,2,False,False,kept
9,active_member,numerical_feature,int64,0,0.0,2,2,False,False,kept


,role,action,column_count
0,categorical_feature,kept,2
1,id,kept,1
2,numerical_feature,kept,8
3,target,kept,1


,role,columns
0,id,[customer_id]
1,categorical,"[country, gender]"
2,numerical,"[credit_score, age, tenure, balance, products_..."
3,target,[estimated_salary]
4,relationship,[estimated_salary]
5,tvd_categorical,"[country, gender]"
6,qi_features,"[country, gender, credit_score, age, tenure, b..."
7,k-anon,"[country, gender, age, tenure, products_number..."
8,passthrough,[]


## Run experiments


In [6]:
suppression_cache = {}
prep_cache = {}
neighbor_cache_store = {}
ranking_rows = []
done_signatures = set()

if SKIP_EXISTING_EXPERIMENTS:
    ranking_rows, done_signatures = load_existing_ranking(EXPERIMENT_RANKING_CSV)
    if done_signatures:
        print(f"Resuming: {len(done_signatures)} configs already in {EXPERIMENT_RANKING_CSV}")

combos = PARAMETER_COMBINATIONS
end = len(combos) if BATCH_LIMIT is None else min(len(combos), BATCH_START + BATCH_LIMIT)
combos = combos.iloc[BATCH_START:end].reset_index(drop=True)
pending = sum(
    1 for _, row in combos.iterrows()
    if config_signature_from_cfg(row_to_config(row)) not in done_signatures
)
print("Grid slice:", len(combos), "configs |", pending, "to run |", len(combos) - pending, "skipped")
if SAVE_EXPERIMENT_ARTIFACTS:
    Path(EXPERIMENT_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print("Experiment artifacts ->", Path(EXPERIMENT_OUTPUT_DIR).resolve())

for i, row in combos.iterrows():
    cfg = row_to_config(row)
    sig = config_signature_from_cfg(cfg)
    if sig in done_signatures:
        print(f"[{i+1}/{len(combos)}] skip (exists) {make_display_name(cfg)}")
        continue

    folder = make_folder_name(len(ranking_rows) + 1, cfg)
    print(f"[{i+1}/{len(combos)}] {folder}")

    tracemalloc.start()
    t0 = time.perf_counter()

    k = int(cfg["k_anonymity"])
    if k not in suppression_cache:
        suppression_cache[k] = identify_suppressed(df, k)
    suppressed, pool_idx, synth_idx = suppression_cache[k]

    scaler = cfg["scaler_method"]
    if scaler not in prep_cache:
        prep_cache[scaler] = fit_preprocessors(df, scaler)
    prep = prep_cache[scaler]

    nkey = config_to_key(cfg)
    if nkey not in neighbor_cache_store:
        neighbor_cache_store[nkey] = build_neighbor_cache(cfg, prep, pool_idx, synth_idx)
    neighbor_cache = neighbor_cache_store[nkey]

    df_out = synthesize_dataset(df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache)
    metrics = compute_metrics(df, df_out, suppressed, RELATIONSHIP_COL)

    runtime_sec = round(time.perf_counter() - t0, 3)
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    if SAVE_EXPERIMENT_ARTIFACTS:
        artifact_dir = save_experiment_outputs(
            folder,
            df_out,
            cfg,
            metrics,
            runtime_sec=runtime_sec,
            peak_memory_mb=round(peak_mem / 1024 / 1024, 2),
            n_suppressed=int(suppressed.sum()),
        )
        print(f"  saved -> {artifact_dir}")

    ranking_rows.append({
        "folder": folder,
        "name": make_display_name(cfg),
        "relationship_col": RELATIONSHIP_COL,
        "target_kind": TARGET_KIND,
        "k_anonymity": k,
        "k_neighbors": int(cfg["k_neighbors"]),
        "cat_gen_method": cfg["cat_gen_method"],
        "num_gen_method": cfg["num_gen_method"],
        "target_gen_method": resolve_target_gen_method(cfg),
        "target_gen_method_config": cfg.get("target_gen_method"),
        "scaler_method": scaler,
        "num_weight": float(cfg["num_weight"]),
        "cat_weight": float(cfg["cat_weight"]),
        "distance_profile": cfg["distance_profile"],
        "distance_mode": cfg["distance_mode"],
        "num_distance_metric": cfg["num_distance_metric"],
        "cat_distance_metric": cfg["cat_distance_metric"],
        "minkowski_p": int(cfg["minkowski_p"]),
        "random_state": int(cfg.get("random_state", RANDOM_STATE)),
        "runtime_sec": runtime_sec,
        "peak_memory_mb": round(peak_mem / 1024 / 1024, 2),
        "n_suppressed": int(suppressed.sum()),
        "tvd_pass_rate": metrics["tvd_pass_rate"],
        "ks_pass_rate": metrics["ks_pass_rate"],
        "mean_tvd": metrics["mean_tvd"],
        "mean_ks": metrics["mean_ks"],
        "mean_corr_drift": metrics["mean_corr_drift"],
        "exact_match_rate": metrics["exact_match_rate"],
        "mean_psi": metrics.get("karabo_summary", {}).get("mean_psi"),
        "target_rate_drift": metrics.get("karabo_summary", {}).get("target_rate_drift"),
        "target_ks": metrics.get("karabo_summary", {}).get("target_ks"),
        "auc_retention_ratio": metrics.get("karabo_summary", {}).get("auc_retention_ratio"),
        "real_auc": metrics.get("karabo_summary", {}).get("real_auc"),
        "synthetic_auc": metrics.get("karabo_summary", {}).get("synthetic_auc"),
        "f1_retention_ratio": metrics.get("karabo_summary", {}).get("f1_retention_ratio"),
        "real_f1": metrics.get("karabo_summary", {}).get("real_f1"),
        "synthetic_f1": metrics.get("karabo_summary", {}).get("synthetic_f1"),
        "r2_retention_ratio": metrics.get("karabo_summary", {}).get("r2_retention_ratio"),
        "real_r2": metrics.get("karabo_summary", {}).get("real_r2"),
        "synthetic_r2": metrics.get("karabo_summary", {}).get("synthetic_r2"),
        "rmse_retention_ratio": metrics.get("karabo_summary", {}).get("rmse_retention_ratio"),
        "real_rmse": metrics.get("karabo_summary", {}).get("real_rmse"),
        "synthetic_rmse": metrics.get("karabo_summary", {}).get("synthetic_rmse"),
        "mean_iv_retention": metrics.get("karabo_summary", {}).get("mean_iv_retention"),
        "overall_pass": metrics["overall_pass"],
    })
    done_signatures.add(sig)

    if len(ranking_rows) % CHECKPOINT_EVERY == 0:
        finalize_ranking(ranking_rows).to_csv(EXPERIMENT_RANKING_CSV, index=False)
        print("  checkpoint ->", EXPERIMENT_RANKING_CSV)

ranking = finalize_ranking(ranking_rows)
ranking.to_csv(EXPERIMENT_RANKING_CSV, index=False)
print("\nSaved", len(ranking), "rows ->", EXPERIMENT_RANKING_CSV)
if len(ranking):
    print("Top:", ranking.iloc[0]["folder"], "| overall_pass=", ranking.iloc[0]["overall_pass"])
ranking.head(10)


Resuming: 36 configs already in experiment_ranking.csv
Grid slice: 48 configs | 12 to run | 36 skipped
Experiment artifacts -> /home/neosoft/KNN_demo/full/experiments
[1/48] skip (exists) k=3 kn=3 cat=mode num=interpolation tgt=interpolation scale=robust weighted_sum/euclidean/hamming/balanced
[2/48] skip (exists) k=3 kn=3 cat=mode num=interpolation tgt=interpolation scale=robust weighted_sum/euclidean/jaccard/balanced
[3/48] skip (exists) k=3 kn=3 cat=mode num=interpolation tgt=interpolation scale=robust weighted_sum/euclidean/cosine/balanced
[4/48] skip (exists) k=3 kn=3 cat=mode num=weighted_mean tgt=weighted_mean scale=robust weighted_sum/euclidean/hamming/balanced
[5/48] skip (exists) k=3 kn=3 cat=mode num=weighted_mean tgt=weighted_mean scale=robust weighted_sum/euclidean/jaccard/balanced
[6/48] skip (exists) k=3 kn=3 cat=mode num=weighted_mean tgt=weighted_mean scale=robust weighted_sum/euclidean/cosine/balanced
[7/48] skip (exists) k=3 kn=3 cat=weighted_mode num=interpolation t

,folder,name,relationship_col,target_kind,k_anonymity,k_neighbors,cat_gen_method,num_gen_method,target_gen_method,target_gen_method_config,...,real_rmse,synthetic_rmse,mean_iv_retention,overall_pass,relationship_score,ks_quality,utility_score,quality_score,composite_score,rank
0,031_k3_kn15_cat-probability_num-interpolation_...,k=3 kn=15 cat=probability num=interpolation tg...,estimated_salary,numeric,3,15,probability,interpolation,interpolation,NaN,...,57559.5957,57560.7341,NaN,True,0.7408,0.721,1.0000,0.9199,0.9057,1
1,007_k3_kn3_cat-weighted_mode_num-interpolation...,k=3 kn=3 cat=weighted_mode num=interpolation t...,estimated_salary,numeric,3,3,weighted_mode,interpolation,interpolation,NaN,...,57559.5957,57560.7341,NaN,True,0.7408,0.721,1.0000,0.9199,0.9043,2
2,019_k3_kn15_cat-mode_num-interpolation_tgt-int...,k=3 kn=15 cat=mode num=interpolation tgt=inter...,estimated_salary,numeric,3,15,mode,interpolation,interpolation,NaN,...,57559.5957,57560.7341,NaN,True,0.7408,0.721,1.0000,0.9199,0.9042,3
3,025_k3_kn15_cat-weighted_mode_num-interpolatio...,k=3 kn=15 cat=weighted_mode num=interpolation ...,estimated_salary,numeric,3,15,weighted_mode,interpolation,interpolation,NaN,...,57559.5957,57560.7341,NaN,True,0.7408,0.721,1.0000,0.9199,0.9042,4
4,013_k3_kn3_cat-probability_num-interpolation_t...,k=3 kn=3 cat=probability num=interpolation tgt...,estimated_salary,numeric,3,3,probability,interpolation,interpolation,NaN,...,57559.5957,57560.7341,NaN,True,0.7408,0.721,1.0000,0.9199,0.9041,5
5,032_k3_kn15_cat-probability_num-interpolation_...,k=3 kn=15 cat=probability num=interpolation tg...,estimated_salary,numeric,3,15,probability,interpolation,interpolation,NaN,...,57559.5957,57576.7001,NaN,True,0.7346,0.710,1.0832,0.9174,0.9030,6
6,008_k3_kn3_cat-weighted_mode_num-interpolation...,k=3 kn=3 cat=weighted_mode num=interpolation t...,estimated_salary,numeric,3,3,weighted_mode,interpolation,interpolation,NaN,...,57559.5957,57576.7001,NaN,True,0.7346,0.710,1.0832,0.9174,0.9022,7
7,026_k3_kn15_cat-weighted_mode_num-interpolatio...,k=3 kn=15 cat=weighted_mode num=interpolation ...,estimated_salary,numeric,3,15,weighted_mode,interpolation,interpolation,NaN,...,57559.5957,57576.7001,NaN,True,0.7346,0.710,1.0832,0.9174,0.9022,8
8,014_k3_kn3_cat-probability_num-interpolation_t...,k=3 kn=3 cat=probability num=interpolation tgt...,estimated_salary,numeric,3,3,probability,interpolation,interpolation,NaN,...,57559.5957,57576.7001,NaN,True,0.7346,0.710,1.0832,0.9174,0.9021,9
9,020_k3_kn15_cat-mode_num-interpolation_tgt-int...,k=3 kn=15 cat=mode num=interpolation tgt=inter...,estimated_salary,numeric,3,15,mode,interpolation,interpolation,NaN,...,57559.5957,57576.7001,NaN,True,0.7346,0.710,1.0832,0.9174,0.9015,10
